In [2]:
from RAG import RAG
from langchain_openai import ChatOpenAI
from ragas import evaluate, EvaluationDataset
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness # Context precision ? 
import os
import datasets

# Load test dataset
test = datasets.load_dataset('m-ric/huggingface_doc_qa_eval', split="train")
# Your test queries
sample_queries = list(test['question'])
# Your test answers 
expected_responses = list(test['answer']) 

# Initialize your RAG pipeline once
rag = RAG()

# Collect evaluation data
dataset = []

for query,reference in zip(sample_queries,expected_responses):

    relevant_docs = rag.get_most_relevant_docs(query)
    response = rag.generate_answer(query, relevant_docs)
    dataset.append(
        {
            "user_input":query,
            "retrieved_contexts":relevant_docs,
            "response":response,
            "reference":reference
        }
    )
# Load into RAGAS
evaluation_dataset = EvaluationDataset.from_list(dataset)

# Configure evaluator LLM pointing at EPFL
evaluator_llm = LangchainLLMWrapper(
    ChatOpenAI(
        model="swiss-ai/Apertus-8B-Instruct-2509",
        base_url="https://inference.rcp.epfl.ch/v1",
        api_key=os.environ["OPENAI_API_KEY"],
    )
)

# Run evaluation
result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness()],
    llm=evaluator_llm,
)
print(result)

/var/folders/20/2kn9rjns09d18w17wc_48kr00000gn/T/ipykernel_79090/3129510224.py:5: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness # Context precision ?
/var/folders/20/2kn9rjns09d18w17wc_48kr00000gn/T/ipykernel_79090/3129510224.py:5: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness # Context precision ?
/var/folders/20/2kn9rjns09d18w17wc_48kr00000gn/T/ipykernel_79090/3129510224.py:5: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. P

Evaluating:   0%|          | 0/130 [00:00<?, ?it/s]

Exception raised in Job[40]: TimeoutError()
Exception raised in Job[103]: TimeoutError()
Exception raised in Job[115]: TimeoutError()
Exception raised in Job[117]: TimeoutError()
Exception raised in Job[118]: TimeoutError()


{'context_recall': 0.8624, 'faithfulness': 0.9348}


In [3]:
result_df = result.to_pandas()
result_df

,user_input,retrieved_contexts,response,reference,context_recall,faithfulness
0,What architecture is the `tokenizers-linux-x64...,[huggingface/tokenizers/blob/main/bindings/nod...,The `tokenizers-linux-x64-musl` binary is desi...,x86_64-unknown-linux-musl,1.000000,1.000000
1,What is the purpose of the BLIP-Diffusion mode...,[huggingface/diffusers/blob/main/docs/source/e...,The BLIP-Diffusion model is a subject-driven i...,The BLIP-Diffusion model is designed for contr...,1.000000,1.000000
2,How can a user claim authorship of a paper on ...,[huggingface/hub-docs/blob/main/docs/hub/paper...,To claim authorship of a paper on the Hugging ...,By clicking their name on the corresponding Pa...,1.000000,0.750000
3,What is the purpose of the /healthcheck endpoi...,[huggingface/datasets-server/blob/main/service...,The purpose of the /healthcheck endpoint in th...,Ensure the app is running,1.000000,1.000000
4,What is the default context window size for Lo...,[huggingface/transformers/blob/main/docs/sourc...,The default context window size for Local Atte...,127 tokens,1.000000,1.000000
...,...,...,...,...,...,...
60,What is the maximum size of a model checkpoint...,[huggingface/transformers/blob/main/docs/sourc...,10GB,10GB,1.000000,1.000000
61,What is the purpose of Weights and Biases (W&B...,[gradio-app/gradio/blob/main/guides/cn/04_inte...,The purpose of Weights and Biases (W&B) for da...,To track their machine learning experiments at...,1.000000,1.000000
62,What is the name of the open-source library cr...,[huggingface/blog/blob/main/optimum-onnxruntim...,The open-source library created by Hugging Fac...,Optimum,1.000000,1.000000
63,What parameter is used to ensure that elements...,[gradio-app/gradio/blob/main/guides/03_buildin...,The parameter is `equal_height`. It is used in...,equal_height,1.000000,1.000000


In [4]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
pd.set_option("display.max_colwidth", None)
import textwrap

df = result.to_pandas()
total = len(df)
current = [0]

output = widgets.Output()

def show_row(idx):
    with output:
        clear_output(wait=True)
        row = df.iloc[idx]
        print(f"{'='*60}")
        print(f"Row {idx + 1} / {total}")
        print(f"{'='*60}")
        print(f"\n📌 QUESTION:")
        print(textwrap.fill(row['user_input'], width=80))
        print(f"\n🤖 LLM ANSWER:")
        print(textwrap.fill(row['response'], width=80))
        print(f"\n✅ REFERENCE:")
        print(textwrap.fill(row['reference'], width=80))
        print(f"\n{'='*60}")

prev_btn = widgets.Button(description="◀ Previous")
next_btn = widgets.Button(description="Next ▶")
buttons = widgets.HBox([prev_btn, next_btn])

def on_prev(b):
    if current[0] > 0:
        current[0] -= 1
    show_row(current[0])

def on_next(b):
    if current[0] < total - 1:
        current[0] += 1
    show_row(current[0])

prev_btn.on_click(on_prev)
next_btn.on_click(on_next)

display(buttons, output)
show_row(0)

Output()